# Resume & Cover-Letter Tailoring Agent -- notebook demo

This notebook is a runnable demo of the **Resume & Cover-Letter Tailoring Agent** project from the course: [`docs/projects/resume-tailor-agent/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/resume-tailor-agent), companion to the fuller local CLI at [`examples/resume-tailor-agent/tailor.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/resume-tailor-agent/tailor.py).

It reads a resume and a job description as text, hands them to a free-tier LLM with an honest-tailoring system prompt, and prints back a match score, a cover-letter draft, and a list of concrete resume edits.

## A note on running this in a notebook

The real version of this tool (`examples/resume-tailor-agent/tailor.py`) reads **your own resume and job descriptions** from files on disk -- that's the whole point of the tool. Colab, Kaggle, and Binder don't have your files.

So **this demo adapts the tool**: it reads the bundled sample resume (`sample_resume.txt`) straight from the course repo, and you paste a real job description below (or fetch one from a URL). That runs every piece of the tool (the file reading, the system prompt, the LLM call, the structured output) honestly -- it's just not pointed at your real resume. **Locally, or in a GitHub Codespace, you'd point it at your own files instead** -- see the [project walkthrough](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/resume-tailor-agent) for that path.

In [ ]:
!pip install -q openai

## Get the sample resume

Fetch the bundled `sample_resume.txt` straight from the course repo -- a fictional data analyst with no machine-learning experience, so it makes an interesting (honest!) run against an ML posting.

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/examples/resume-tailor-agent/sample_resume.txt"
resume = urllib.request.urlopen(url).read().decode("utf-8")
print(f"Loaded sample resume: {len(resume)} characters")

## Get a job description

Fetch the bundled `sample_job.txt` straight from the course repo -- a fictional
*Junior Machine Learning Engineer* posting, so paired with the sample data
analyst resume above it makes an interesting (honest!) run. To use a real
posting instead, replace the `url` below with a plain-text URL, or paste the
text of a job description into the `job` string.

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/examples/resume-tailor-agent/sample_job.txt"
job = urllib.request.urlopen(url).read().decode("utf-8")
print(f"Loaded sample job description: {len(job)} characters")

## The tailoring system prompt

This is the exact `SYSTEM_PROMPT` from `tailor.py` -- it's what turns a general-purpose chat model into a strict, honest tailoring assistant. The no-fabrication rule is the whole point: never invent skills or titles, never upgrade a bullet point, report gaps instead of hiding them.

In [ ]:
SYSTEM_PROMPT = """\
You are a meticulous, honest resume-and-cover-letter tailoring assistant.

You will be given a RESUME and a JOB DESCRIPTION. Your job is to help the
candidate apply for THIS job, using ONLY facts that already exist in their
resume. This is non-negotiable:

- NEVER invent skills, technologies, tools, titles, employers, projects,
  dates, numbers, or credentials that are not already on the resume.
- NEVER reword an existing bullet point into something that is not plainly
  supported by it. Rephrase and re-order freely, but do not upgrade.
- If the resume is missing something the job clearly asks for, say so in
  the resume edits list instead of pretending the candidate has it.

Produce exactly three sections:

1. MATCH SCORE: A number from 0-100 with a two-sentence rationale. Be
  honest -- an 82 with a clear explanation beats a 95 that can't be backed
  up by the resume.

2. COVER LETTER DRAFT: A complete, ready-to-edit cover letter of 2-3 short
  paragraphs, addressed to a hiring manager, that connects specific items
  already on the resume to the specific requirements of THIS job. Every
  claim it makes must trace back to the resume.

3. RESUME EDITS: A numbered list of concrete, actionable changes to make
  to the resume for this job -- reordering bullets, swapping which
  projects get highlighted, removing irrelevant lines, adding keywords
  that genuinely match existing experience. Each edit states what to
  change and why. Where the resume genuinely lacks something the job
  wants, state that plainly as a gap, never as a fake achievement.

Be specific and concrete throughout. Do not pad. Do not flatter.
"""

## Get a free-tier API key

This demo defaults to **GitHub Models** -- free, no separate signup, just a personal access token with the `models: read` scope from [github.com/settings/tokens](https://github.com/settings/tokens). Any of the other five providers wired up in `tailor.py` (Gemini, Groq, Mistral, Cerebras, OpenRouter) work too -- see that file's `PROVIDERS` dict for their base URLs and env var names, and adjust `LLM_PROVIDER` below.

The key is entered with `getpass` so it never gets typed into a visible cell or saved into this notebook's output -- never hardcode a real API key here.

In [ ]:
import os
from getpass import getpass

LLM_PROVIDER = "github"  # change to gemini / groq / mistral / cerebras / openrouter if you prefer
os.environ["GITHUB_TOKEN"] = getpass("Enter your free-tier GitHub Models token (GITHUB_TOKEN): ")

## The tailoring logic itself

This mirrors `truncate`, `PROVIDERS`, and `tailor` from `tailor.py` directly -- the same truncation cap, the same free-tier providers (all exposed through the `openai` client, just pointed at each provider's own OpenAI-compatible endpoint), and the same call shape.

In [ ]:
from openai import OpenAI

MAX_TEXT_CHARS = 30_000


def truncate(text: str, max_chars: int = MAX_TEXT_CHARS) -> str:
    """Cuts an oversized document down to a size that fits a free-tier context window."""
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + f"\n\n... [truncated -- {len(text) - max_chars} more characters not shown] ..."


def _build_github_client() -> OpenAI:
    return OpenAI(api_key=os.environ["GITHUB_TOKEN"], base_url="https://models.github.ai/inference")


def _build_gemini_client() -> OpenAI:
    return OpenAI(
        api_key=os.environ["GOOGLE_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )


def _build_groq_client() -> OpenAI:
    return OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")


def _build_mistral_client() -> OpenAI:
    return OpenAI(api_key=os.environ["MISTRAL_API_KEY"], base_url="https://api.mistral.ai/v1")


def _build_cerebras_client() -> OpenAI:
    return OpenAI(api_key=os.environ["CEREBRAS_API_KEY"], base_url="https://api.cerebras.ai/v1")


def _build_openrouter_client() -> OpenAI:
    return OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")


PROVIDERS = {
    "github": (_build_github_client, "gpt-4o-mini"),
    "gemini": (_build_gemini_client, "gemini-3.5-flash"),
    "groq": (_build_groq_client, "llama-3.3-70b-versatile"),
    "mistral": (_build_mistral_client, "mistral-small-latest"),
    "cerebras": (_build_cerebras_client, "llama-3.3-70b"),
    "openrouter": (_build_openrouter_client, "meta-llama/llama-3.3-70b-instruct:free"),
}


def tailor(resume: str, job: str, provider: str = LLM_PROVIDER) -> str:
    """Sends a resume + job description to a free-tier LLM and returns the tailored result."""
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown provider '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    build_client, model = PROVIDERS[provider]
    client = build_client()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"RESUME:\n```\n{truncate(resume)}\n```\n\nJOB DESCRIPTION:\n```\n{truncate(job)}\n```",
            },
        ],
    )
    return response.choices[0].message.content

## Run the tailoring

Using the bundled sample resume and the job description from above -- every section the prompt asks for (match score, cover letter draft, resume edits) should print. Then do the **no-fabrication audit** from the lesson: read the draft claim by claim against the resume, and check nothing was invented.

In [ ]:
print(f"Tailoring a {len(resume)}-char resume against a {len(job)}-char job description...\n")
result = tailor(resume, job)
print(result)